# Compare local and Colab benchmark results

Run `python -m ailab.benchmark` on your computer and `python -m ailab.benchmark --environment colab` in the Colab copy of this project. Save both JSON files under `results/`, then run these notebook cells from the project root.

This report compares NumPy matrix multiplication timings. The benchmark does not select a GPU, so the environment labels identify where you ran it; they do not say which hardware performed the calculation. Compare runs only when their matrix size and data type match.

In [ ]:
import json
from pathlib import Path


def load_result(path):
    if not path.exists():
        print(f"No result file found at {path}. Run the benchmark for this environment first.")
        return None

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        print(f"The result file at {path} is not valid JSON. Run the benchmark again.")
        return None


local_result = load_result(Path("results/local_benchmark.json"))
colab_result = load_result(Path("results/colab_benchmark.json"))

In [ ]:
if local_result is None or colab_result is None:
    print("Add both benchmark result files before comparing.")
elif (local_result["matrix_size"], local_result["dtype"]) != (
    colab_result["matrix_size"], colab_result["dtype"]
):
    print("These runs use different matrix sizes or number formats, so their timings are not directly comparable.")
    print(f"Local: matrix size {local_result['matrix_size']}, {local_result['dtype']}")
    print(f"Colab: matrix size {colab_result['matrix_size']}, {colab_result['dtype']}")
else:
    local_seconds = local_result["average_seconds"]
    colab_seconds = colab_result["average_seconds"]
    print(f"Matrix size: {local_result['matrix_size']} x {local_result['matrix_size']}")
    print(f"Number format: {local_result['dtype']}")
    print(f"Local average: {local_seconds:.4f} seconds")
    print(f"Colab average: {colab_seconds:.4f} seconds")

    if local_seconds == 0 or colab_seconds == 0:
        print("A run took too little measurable time to calculate a timing ratio. Try a larger matrix size.")
    elif local_seconds > colab_seconds:
        print(f"Colab was {local_seconds / colab_seconds:.2f}x faster in this run.")
    elif colab_seconds > local_seconds:
        print(f"Local was {colab_seconds / local_seconds:.2f}x faster in this run.")
    else:
        print("The average times were equal in this run.")